# JMIR evidence-selection experiment — evaluation casebook draft

Creates a checkpointed multilingual draft for the retained 100 VQA-RAD + 25 SLAKE evaluation cases. It does **not** approve or freeze cases. English propositions are drafted by an independent text model (`llama3.2:3b`), while questions and propositions are translated with NLLB-200 into Indonesian and Malay.

## 1. Mount Drive and locate the candidate casebook
Run this in the same T4 session used for Notebook 03 when possible.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, subprocess, time, urllib.request
import pandas as pd

ROOT=Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK_DIR=ROOT/'casebooks'; CASEBOOK_DIR.mkdir(parents=True,exist_ok=True)
matches=[p for p in ROOT.rglob('candidate_casebook*.csv') if 'frozen' not in p.name.lower() and 'machine_draft' not in p.name.lower()]
if not matches: raise FileNotFoundError('No candidate_casebook CSV found under ROOT')
for i,p in enumerate(matches): print(i,p)
SOURCE=max(matches,key=lambda p:p.stat().st_mtime)
print('Using:',SOURCE)
df=pd.read_csv(SOURCE,dtype=str,keep_default_na=False)
assert len(df)==135 and df.phase.value_counts().to_dict()=={'evaluation':125,'pilot':10}
assert df[df.phase.eq('evaluation')].dataset.value_counts().to_dict()=={'vqa_rad':100,'slake':25}
source_sha=hashlib.sha256(SOURCE.read_bytes()).hexdigest()
print('Source SHA256:',source_sha)

Mounted at /content/drive
0 /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook.csv
Using: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook.csv
Source SHA256: fb2f677350d2f9b35c871cca12f2153bceb2f992ab1f344cab31633242ac7d40


## 2. Install the current Ollama base package and GPU payload
This installer uses exact release-asset names and excludes MLX/ROCm packages.

In [3]:
api='https://api.github.com/repos/ollama/ollama/releases/latest'
req=urllib.request.Request(api,headers={'Accept':'application/vnd.github+json','User-Agent':'JMIR-Colab-GPU-Benchmark'})
with urllib.request.urlopen(req,timeout=60) as response: release=json.loads(response.read())
assets={a['name']:a['browser_download_url'] for a in release.get('assets',[])}
base_name='ollama-linux-amd64.tar.zst'
assert base_name in assets,f'{base_name} unavailable. Linux assets: {[n for n in assets if "linux-amd64" in n]}'
cuda_names=[n for n in assets if 'linux-amd64' in n and 'cuda' in n.lower() and n.endswith('.tar.zst')]
print('Ollama release:',release['tag_name'])
print('CUDA payload candidates:',cuda_names or 'none; base package will be tested')

subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','zstd'],check=True)

def download_extract(name):
    target=Path('/content')/name
    print('Downloading',name)
    subprocess.run(['curl','-fL','--retry','3','--retry-all-errors','-o',str(target),assets[name]],check=True)
    print('Extracting',name,round(target.stat().st_size/1024**2,1),'MB')
    subprocess.run(['tar','--use-compress-program=unzstd','-xf',str(target),'-C','/usr'],check=True)
    target.unlink(missing_ok=True)

if subprocess.run(['bash','-lc','command -v ollama'],capture_output=True).returncode!=0:
    download_extract(base_name)
    # Prefer a CUDA 12 payload for Colab T4; otherwise use the first generic CUDA payload.
    if cuda_names:
        cuda_name=next((n for n in cuda_names if 'v12' in n.lower()),cuda_names[0])
        download_extract(cuda_name)

ollama_binary=next((str(p) for p in [Path('/usr/bin/ollama'),Path('/usr/local/bin/ollama')] if p.exists()),None)
assert ollama_binary,'Ollama executable not found after extraction'
print(subprocess.run([ollama_binary,'--version'],capture_output=True,text=True).stdout.strip())

Ollama release: v0.34.2
CUDA payload candidates: none; base package will be tested
Extracting ollama-linux-amd64.tar.zst 1361.4 MB


In [4]:
def ollama_alive():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2)
        return True
    except Exception:return False

if not ollama_alive():
    log_handle=open('/content/ollama_gpu_server.log','a')
    env=os.environ.copy(); env['OLLAMA_HOST']='127.0.0.1:11434'
    ollama_process=subprocess.Popen([ollama_binary,'serve'],stdout=log_handle,stderr=subprocess.STDOUT,env=env)
    for _ in range(90):
        if ollama_alive():break
        time.sleep(1)
assert ollama_alive(),Path('/content/ollama_gpu_server.log').read_text(errors='replace')
print('Ollama server ready.')
print(Path('/content/ollama_gpu_server.log').read_text(errors='replace')[-3000:])

Ollama server ready.
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKbz5o/YL5gA115Yzs6vm3Hx11776I089SBOZ5VzPXem

time=2026-09-21T11:20:37.852Z level=INFO source=routes.go:1940 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_N

## 2. Connect to Ollama and pull the independent proposition model
Notebook 03 should already have started Ollama. If this cell says the server is unavailable, rerun Sections 2–3 of Notebook 03 first.

In [6]:
def api_json(path,payload=None,timeout=900):
    data=None if payload is None else json.dumps(payload).encode()
    req=urllib.request.Request('http://127.0.0.1:11434'+path,data=data,headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req,timeout=timeout) as response:return json.loads(response.read())
try: api_json('/api/tags')
except Exception as e: raise RuntimeError('Ollama is not running. Rerun Notebook 03 installation/server cells.') from e
OLLAMA=next((p for p in ['/usr/bin/ollama','/usr/local/bin/ollama'] if Path(p).exists()),'ollama')
DRAFT_MODEL='llama3.2:3b'
subprocess.run([OLLAMA,'pull',DRAFT_MODEL],check=True)
print('Drafting model ready:',DRAFT_MODEL)

Drafting model ready: llama3.2:3b


## 3. Draft English evidence propositions with checkpoints
A proposition must be a neutral noun phrase that fits after “Evidence of … is present/absent.” Open-ended or nonbinary questions are flagged instead of repaired.

In [7]:
PROP_CHECKPOINT=CASEBOOK_DIR/'evaluation_proposition_draft_checkpoint.jsonl'
SYSTEM='''You prepare stimuli for a medical visual question-answering study. Given one English question, decide whether it is genuinely answerable by Yes or No. If suitable, return a concise neutral English noun phrase naming exactly the finding asked about. It must fit grammatically after: Evidence of {phrase} is present. Do not include yes, no, present, absent, detected, or evidence of. Preserve laterality and anatomy. Return strict JSON only with keys suitable (boolean), proposition (string), and reason (string).'''
def append_jsonl(path,obj):
    with open(path,'a',encoding='utf-8') as f:
        f.write(json.dumps(obj,ensure_ascii=False)+'\n');f.flush();os.fsync(f.fileno())
def existing_records(path):
    out={}
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                x=json.loads(line);out[x['case_id']]=x
            except:pass
    return out
def parse_json_object(text):
    m=re.search(r'\{.*\}',text,re.S)
    if not m: raise ValueError('No JSON object: '+text[:200])
    return json.loads(m.group(0))
records=existing_records(PROP_CHECKPOINT)
evaluation=df[df.phase.eq('evaluation')]
for n,(_,row) in enumerate(evaluation.iterrows(),1):
    if row.case_id in records and records[row.case_id].get('status')=='ok':
        print(f'[{n}/125] skip',row.case_id);continue
    # Deterministic safeguard: an explicit WH-question is not binary even if its dataset answer is yes/no.
    if re.match(r'^\s*(what|which|who|where|when|why|how)\b',row.question_en,re.I):
        rec={'case_id':row.case_id,'status':'ok','suitable':False,'proposition':'','reason':'Open-ended WH-question is incompatible with binary A/B options.','model':'structural_rule','timestamp_utc':datetime.now(timezone.utc).isoformat()}
    else:
        payload={'model':DRAFT_MODEL,'stream':False,'format':'json','messages':[{'role':'system','content':SYSTEM},{'role':'user','content':row.question_en}],'options':{'temperature':0,'seed':20260921,'num_predict':120}}
        start=time.time()
        try:
            response=api_json('/api/chat',payload); parsed=parse_json_object(response['message']['content'])
            rec={'case_id':row.case_id,'status':'ok','suitable':bool(parsed.get('suitable')),'proposition':str(parsed.get('proposition','')).strip().rstrip('.'),'reason':str(parsed.get('reason','')).strip(),'model':DRAFT_MODEL,'elapsed_seconds':round(time.time()-start,3),'timestamp_utc':datetime.now(timezone.utc).isoformat()}
        except Exception as e: rec={'case_id':row.case_id,'status':'error','error':repr(e),'timestamp_utc':datetime.now(timezone.utc).isoformat()}
    append_jsonl(PROP_CHECKPOINT,rec);records[row.case_id]=rec
    print(f'[{n}/125]',row.case_id,rec.get('suitable'),rec.get('proposition',rec.get('error')))
assert len([x for x in records.values() if x.get('status')=='ok'])>=125,'Rerun this cell to retry incomplete cases'
print('Proposition drafting complete.')

[1/125] C011 True enhancing
[2/125] C012 True pneumothorax
[3/125] C013 True nodules
[4/125] C014 True ventricles
[5/125] C015 True normal
[6/125] C016 True sides
[7/125] C017 True Wedge
[8/125] C018 True pleural effusion
[9/125] C019 True abnormality
[10/125] C020 True cerebral edema
[11/125] C021 True fracture
[12/125] C022 True contrast
[13/125] C023 False 
[14/125] C024 True septations
[15/125] C025 True Hypodense lesions
[16/125] C026 True mass
[17/125] C027 True liver
[18/125] C028 True ectatic or aneurysmal
[19/125] C029 False fluid
[20/125] C030 True ventricles
[21/125] C031 True pulmonary edema
[22/125] C032 True symmetry
[23/125] C033 True sulci
[24/125] C034 True skull fracture
[25/125] C035 True right kidney
[26/125] C036 True skull fracture
[27/125] C037 True optic nerve
[28/125] C038 False underexposed
[29/125] C039 True pulmonary abnormalities
[30/125] C040 True clavicle fracture
[31/125] C041 True pleural effusion
[32/125] C042 False atrophied
[33/125] C043 False diagno

## 4. Inspect validity decisions before translation
C023 is expected to be flagged because its source question is open-ended despite carrying a binary answer.

In [8]:
prop_df=pd.DataFrame(records.values()).sort_values('case_id')
display(prop_df[prop_df.status.eq('ok') & ~prop_df.suitable][['case_id','reason','model']])
display(prop_df[prop_df.status.eq('ok') & prop_df.suitable][['case_id','proposition']].head(15))
print('Suitable:',int(prop_df.suitable.eq(True).sum()),'Flagged:',int(prop_df.suitable.eq(False).sum()))

,case_id,reason,model
12,C023,Open-ended WH-question is incompatible with bi...,structural_rule
18,C029,Fluid accumulation in the anterior abdominal w...,llama3.2:3b
27,C038,The question is not answerable with a simple Y...,llama3.2:3b
31,C042,The question is not answerable with a simple Y...,llama3.2:3b
32,C043,insufficient information,llama3.2:3b
39,C050,"The trachea is not midline, it is located off ...",llama3.2:3b
40,C051,The question is asking for a binary classifica...,llama3.2:3b
43,C054,"Lungs are typically affected by consolidation,...",llama3.2:3b
47,C058,The question is not answerable with a simple Y...,llama3.2:3b
51,C062,The question is not answerable by Yes or No be...,llama3.2:3b


,case_id,proposition
0,C011,enhancing
1,C012,pneumothorax
2,C013,nodules
3,C014,ventricles
4,C015,normal
5,C016,sides
6,C017,Wedge
7,C018,pleural effusion
8,C019,abnormality
9,C020,cerebral edema


Suitable: 96 Flagged: 29


## 5. Install NLLB and translate questions/propositions
NLLB is independent of the two evaluated VLMs. Translation checkpoints are saved after each batch.

In [9]:
subprocess.run(['pip','-q','install','transformers>=4.46,<5','sentencepiece','sacremoses'],check=True)
# Release the proposition model before loading NLLB.
try: api_json('/api/generate',{'model':DRAFT_MODEL,'keep_alive':0})
except:pass
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
NLLB='facebook/nllb-200-distilled-600M'
tokenizer=AutoTokenizer.from_pretrained(NLLB,src_lang='eng_Latn')
mt=AutoModelForSeq2SeqLM.from_pretrained(NLLB).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
print('NLLB device:',next(mt.parameters()).device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

NLLB device: cuda:0


In [10]:
TRANSLATION_CHECKPOINT=CASEBOOK_DIR/'evaluation_translation_checkpoint.jsonl'
translations=existing_records(TRANSLATION_CHECKPOINT)
prop_map={k:v for k,v in records.items() if v.get('status')=='ok'}
def translate_batch(texts,target,batch_size=16):
    outputs=[]
    for start in range(0,len(texts),batch_size):
        batch=texts[start:start+batch_size]
        enc=tokenizer(batch,return_tensors='pt',padding=True,truncation=True,max_length=128).to(next(mt.parameters()).device)
        with torch.inference_mode(): gen=mt.generate(**enc,forced_bos_token_id=tokenizer.convert_tokens_to_ids(target),max_new_tokens=128,num_beams=4)
        outputs.extend(tokenizer.batch_decode(gen,skip_special_tokens=True))
    return outputs
pending=[r for _,r in evaluation.iterrows() if r.case_id not in translations]
for start in range(0,len(pending),16):
    batch=pending[start:start+16]
    valid=[r for r in batch if prop_map[r.case_id].get('suitable')]
    if valid:
        questions=[r.question_en.strip() for r in valid]; props=[prop_map[r.case_id]['proposition'] for r in valid]
        q_id=translate_batch(questions,'ind_Latn');q_ms=translate_batch(questions,'zsm_Latn')
        p_id=translate_batch(props,'ind_Latn');p_ms=translate_batch(props,'zsm_Latn')
        for r,a,b,c,d in zip(valid,q_id,q_ms,p_id,p_ms):
            rec={'case_id':r.case_id,'status':'ok','question_id':a,'question_ms':b,'proposition_id':c,'proposition_ms':d,'translation_model':NLLB,'timestamp_utc':datetime.now(timezone.utc).isoformat()}
            append_jsonl(TRANSLATION_CHECKPOINT,rec);translations[r.case_id]=rec
    for r in batch:
        if not prop_map[r.case_id].get('suitable'):
            rec={'case_id':r.case_id,'status':'not_translated','reason':prop_map[r.case_id]['reason'],'timestamp_utc':datetime.now(timezone.utc).isoformat()}
            append_jsonl(TRANSLATION_CHECKPOINT,rec);translations[r.case_id]=rec
    print('Checkpointed through batch starting',start+1)
print('Translation records:',len(translations))

Checkpointed through batch starting 1
Checkpointed through batch starting 17
Checkpointed through batch starting 33
Checkpointed through batch starting 49
Checkpointed through batch starting 65
Checkpointed through batch starting 81
Checkpointed through batch starting 97
Checkpointed through batch starting 113
Translation records: 125


## 6. Build the machine-drafted casebook
Valid rows remain `machine_drafted`, not `approved`. Flagged rows become `replacement_required`. Pilot rows are preserved unchanged.

In [11]:
draft=df.copy()
for idx,row in draft[draft.phase.eq('evaluation')].iterrows():
    p=prop_map[row.case_id];t=translations[row.case_id]
    if p.get('suitable') and t.get('status')=='ok':
        draft.at[idx,'proposition_en']=p['proposition']
        for col in ['question_id','question_ms','proposition_id','proposition_ms']:draft.at[idx,col]=t[col]
        draft.at[idx,'clinical_validity_notes']='Structurally binary; machine-drafted proposition and NLLB translations require final linguistic review.'
        draft.at[idx,'include_after_review']=''
        draft.at[idx,'reviewer_initials']=''
        draft.at[idx,'review_status']='machine_drafted'
    else:
        draft.at[idx,'clinical_validity_notes']=p.get('reason','Drafting or translation failed.')
        draft.at[idx,'include_after_review']='no'
        draft.at[idx,'reviewer_initials']=''
        draft.at[idx,'review_status']='replacement_required'
OUTPUT=CASEBOOK_DIR/'candidate_casebook_evaluation_machine_draft.csv'
draft.to_csv(OUTPUT,index=False,encoding='utf-8')
sha=hashlib.sha256(OUTPUT.read_bytes()).hexdigest()
(OUTPUT.with_suffix(OUTPUT.suffix+'.sha256')).write_text(sha+'  '+OUTPUT.name+'\n')
print('Saved:',OUTPUT)
print('SHA256:',sha)
print('Statuses:',draft.review_status.value_counts().to_dict())
for cols in [['question_id','question_ms','proposition_en','proposition_id','proposition_ms']]:
    print('Missing among machine drafts:',{c:int((draft.loc[draft.review_status.eq('machine_drafted'),c].str.strip()=='').sum()) for c in cols})
display(draft.loc[draft.phase.eq('evaluation'),['case_id','dataset','question_en','question_id','question_ms','proposition_en','proposition_id','proposition_ms','review_status','clinical_validity_notes']].head(10))

Saved: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/casebooks/candidate_casebook_evaluation_machine_draft.csv
SHA256: 9e2b3a6156271542fb37af508684d5ba213e1059e53460d57431412e6df8da5c
Statuses: {'machine_drafted': 96, 'replacement_required': 29, 'approved': 10}
Missing among machine drafts: {'question_id': 0, 'question_ms': 0, 'proposition_en': 0, 'proposition_id': 0, 'proposition_ms': 0}


,case_id,dataset,question_en,question_id,question_ms,proposition_en,proposition_id,proposition_ms,review_status,clinical_validity_notes
10,C011,vqa_rad,is the lesion enhancing?,Apakah lesi meningkat?,Adakah kecederaan meningkat?,enhancing,meningkatkan,meningkatkan,machine_drafted,Structurally binary; machine-drafted propositi...
11,C012,vqa_rad,is a pneuomothorax present?,Pneumothorax hadir?,Adakah terdapat pneumothorax?,pneumothorax,pneumothorax,pneumothorax,machine_drafted,Structurally binary; machine-drafted propositi...
12,C013,vqa_rad,are these small opacities in the right lung ca...,Apakah ini adalah opacities kecil di kalsifika...,Adakah ini ketidakselesaan kecil dalam kalsifi...,nodules,Nodule,nodule,machine_drafted,Structurally binary; machine-drafted propositi...
13,C014,vqa_rad,are the ventricles enlarged?,Apakah ventrikel membesar?,Adakah ventrikel membesar?,ventricles,ventrikel,ventrikel,machine_drafted,Structurally binary; machine-drafted propositi...
14,C015,vqa_rad,is this a normal x ray?,Apakah ini sinar-X normal?,Adakah ini sinar-X biasa?,normal,normal,normal,machine_drafted,Structurally binary; machine-drafted propositi...
15,C016,vqa_rad,were both sides affected?,Apakah kedua belah pihak terpengaruh?,Adakah kedua-dua pihak terjejas?,sides,sisi,sisi,machine_drafted,Structurally binary; machine-drafted propositi...
16,C017,vqa_rad,is the lesion wedge-shaped?,Apakah lesi berbentuk kepiting?,Adakah kecederaan berbentuk wedge?,Wedge,Wedge,Wedge,machine_drafted,Structurally binary; machine-drafted propositi...
17,C018,vqa_rad,is a pleural effusion present?,Apakah ada pembuluh darah pleural?,Adakah peluncuran pleural hadir?,pleural effusion,pembuangan pleural,pembekuan pleural,machine_drafted,Structurally binary; machine-drafted propositi...
18,C019,vqa_rad,is this picture abnormal?,Apakah gambar ini tidak normal?,Adakah gambar ini tidak normal?,abnormality,abnormalitas,kelainan,machine_drafted,Structurally binary; machine-drafted propositi...
19,C020,vqa_rad,do we see diffuse cerebral edema in this mri?,Apakah kita melihat edema otak yang difus dala...,Adakah kita melihat edema otak yang difus dala...,cerebral edema,edema otak,edema otak,machine_drafted,Structurally binary; machine-drafted propositi...


## Stop here
Upload `candidate_casebook_evaluation_machine_draft.csv` from the project `casebooks` folder. It needs linguistic/semantic review and replacement of every `replacement_required` row before evaluation freezing.